# Rule 规则引擎使用指南

本 Notebook 演示 `Rule` 类及相关方法的使用，包括：
- 创建和解析规则
- 规则匹配与过滤
- 规则评估与报告
- 规则组合与逻辑运算
- 规则集分析

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from hscredit.core.rules import Rule
from hscredit.report.ruleset_report import ruleset_report

## 1. 创建示例数据

In [ ]:
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    'score': np.random.randint(300, 800, n_samples),
    'age': np.random.randint(18, 65, n_samples),
    'income': np.random.randint(3000, 50000, n_samples),
    'target': np.random.binomial(1, 0.1, n_samples),
    'loan_amount': np.random.randint(10000, 100000, n_samples),
    'overdue_days': np.random.choice([0, 3, 7, 15, 30, 60, 90], n_samples, p=[0.8, 0.05, 0.05, 0.03, 0.03, 0.02, 0.02])
})

print(f"数据集样本数: {len(data)}")
print(f"坏样本率: {data['target'].mean():.2%}")
data.head()

## 2. Rule 基础用法

### 2.1 创建规则

In [ ]:
# 从字符串创建规则
rule1 = Rule('score < 500')
print(f"规则1: {rule1}")
print(f"规则1涉及的字段: {rule1.feature_names_in_}")

rule2 = Rule('age < 25')
print(f"\n规则2: {rule2}")

# 复杂规则（多条件）
rule3 = Rule('score < 500 and income < 10000')
print(f"\n规则3: {rule3}")
print(f"规则3涉及的字段: {rule3.feature_names_in_}")

### 2.2 规则匹配 (predict)

In [ ]:
# 预测：返回每条记录是否命中规则
predictions = rule1.predict(data)
print(f"规则命中情况: {predictions.sum()} / {len(predictions)}")
print(f"命中率: {predictions.mean():.2%}")

# 查看前10条
print("\n前10条数据命中情况:")
data.head(10).assign(命中规则1=predictions.head(10))

### 2.3 规则过滤 (filter)

In [ ]:
# 过滤：返回满足规则的数据子集
filtered = rule1.filter(data)
print(f"原始数据量: {len(data)}")
print(f"命中规则数据量: {len(filtered)}")
print(f"坏样本率: {filtered['target'].mean():.2%}")
filtered.head()

## 3. 规则报告 (report)

### 3.1 基础规则报告

In [ ]:
# 生成规则报告
report = rule1.report(data, target='target')
print(f"报告列名: {list(report.columns)}")
report

### 3.2 带描述的规则报告

In [ ]:
# 使用 desc 参数添加规则描述
report_with_desc = rule1.report(data, target='target', desc='低分规则')
report_with_desc[['规则分类', '指标名称', '指标含义', '分箱', '样本总数', '坏样本率', 'LIFT值']]

### 3.3 金额口径规则报告

In [ ]:
# 使用 amount 参数进行金额口径分析
report_amount = rule1.report(data, target='target', amount='loan_amount', desc='低分规则-金额口径')
report_amount[['分箱', '样本总数', '坏样本数', '坏样本率', 'LIFT值']]

### 3.4 使用逾期天数生成报告（多标签）

In [ ]:
# 使用 overdue 和 dpds 参数进行多标签分析
report_overdue = rule1.report(
    data, 
    overdue='overdue_days', 
    dpds=[7, 15],  # DPD7+ 和 DPD15+
    desc='低分规则'
)
print(f"报告形状: {report_overdue.shape}")
print(f"列结构: {report_overdue.columns.tolist()[:5]}...")
report_overdue.head(4)

## 4. 规则组合与逻辑运算

### 4.1 规则与 (AND)

In [ ]:
# 规则与：同时满足两个条件
combined_and = rule1 & rule2
print(f"规则1: {rule1}")
print(f"规则2: {rule2}")
print(f"组合规则 (AND): {combined_and}")

report_and = combined_and.report(data, target='target')
report_and[['分箱', '样本总数', '坏样本率', 'LIFT值']]

### 4.2 规则或 (OR)

In [ ]:
# 规则或：满足任一条件
combined_or = rule1 | rule2
print(f"组合规则 (OR): {combined_or}")

report_or = combined_or.report(data, target='target')
report_or[['分箱', '样本总数', '坏样本率', 'LIFT值']]

### 4.3 规则非 (NOT)

In [ ]:
# 规则非：取反
combined_not = ~rule1
print(f"原规则: {rule1}")
print(f"取反规则 (NOT): {combined_not}")

report_not = combined_not.report(data, target='target')
report_not[['分箱', '样本总数', '坏样本率', 'LIFT值']]

### 4.4 复杂规则组合

In [ ]:
# 复杂组合：(低分 且 年轻人) 或 低收入
rule_low_income = Rule('income < 5000')
complex_rule = (rule1 & rule2) | rule_low_income
print(f"复杂组合规则: {complex_rule}")

report_complex = complex_rule.report(data, target='target', desc='复杂组合规则')
report_complex[['分箱', '样本总数', '坏样本率', 'LIFT值']]

## 5. 规则集分析 (ruleset_report)

### 5.1 基础规则集报告

In [ ]:
# 创建多个规则
rules = [
    Rule('score < 400'),
    Rule('age < 22'),
    Rule('income < 5000')
]

print("规则列表:")
for i, r in enumerate(rules, 1):
    print(f"  {i}. {r}")

# 生成规则集报告
ruleset_report_df = ruleset_report(data, rules=rules, target='target')
ruleset_report_df[['分箱', '样本总数', '好样本数', '坏样本数', '坏样本率', 'LIFT值']]

### 5.2 金额口径规则集报告

In [ ]:
# 金额口径规则集报告
ruleset_report_amount = ruleset_report(
    data, 
    rules=rules, 
    target='target',
    amount='loan_amount'
)
ruleset_report_amount[['分箱', '样本总数', '坏样本数', '坏样本率']]

### 5.3 多标签规则集报告

In [ ]:
# 多标签规则集报告
ruleset_report_overdue = ruleset_report(
    data, 
    rules=rules, 
    overdue='overdue_days',
    dpds=[7, 15],
    desc='多规则组合'
)
print(f"报告形状: {ruleset_report_overdue.shape}")
print(f"列层级数: {ruleset_report_overdue.columns.nlevels}")
ruleset_report_overdue.head(6)

## 6. 完整报告示例

In [ ]:
# 展示完整的报告信息
final_report = rule1.report(data, target='target', desc='低分规则演示')
final_report

## 7. 总结

**Rule 类主要功能**：
- `Rule(expr)`: 从字符串表达式创建规则
- `predict(data)`: 预测数据是否命中规则
- `filter(data)`: 过滤命中规则的数据
- `report()`: 生成规则效果报告
- `&` `|` `~`: 规则组合运算（与、或、非）

**ruleset_report 功能**：
- 评估多个规则的综合效果
- 支持样本口径和金额口径
- 支持多标签分析（不同逾期天数定义）

**报告列说明**：
- `样本总数/占比`: 样本数量/占比（金额口径下为金额汇总）
- `好/坏样本数`: 好/坏样本数量
- `坏样本率`: 坏样本占比
- `LIFT值`: 规则提升度
- `风险拒绝比`: 坏账改善/样本占比
- `准确率/精确率/召回率/F1分数`: 分类指标